# DNN — backtest

Forecasts are saved under `results/dnn/test/full/<model>.csv`.

Backtest of the DNN on the test set, every day of it (364 days). The model is the same as in `03_model_selection/02_DNN.ipynb`: the fully connected network from `epftoolbox` with two hidden layers, trained with the MAE loss, which forecasts all 24 hours of the next day at once. `calibration_window` is in years of 364 days, so `2` and `3` are the same history as LEAR's 728 and 1092 days.

The hyperparameters were tuned on the training set in `02_DNN.ipynb` (the trials files are in `data/dnn/hyperparameters/`) and are used here unchanged, so there is no tuning in this notebook. Every test day the network is retrained with them on the last `calibration_window` years, and a random quarter of the training weeks is held out for early stopping.

The model needs history before every test day, so the forecasts are made on `dataset_full`, the train, validation and test sets joined.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent.parent))

from src.config import EPF_EXOG
from src.utils.epf import evaluate_dnn, forecast_path
from src.utils.evaluation import compare_models, load_forecasts
from src.utils.file_io import load_final_dataset

/opt/homebrew/Caskroom/miniforge/base/envs/epf-lear-dnn/lib/python3.10/site-packages/hyperopt/atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
dataset_full = load_final_dataset("epf", "pl", "full")
dataset_test = load_final_dataset("epf", "pl", "test")
prices = dataset_full["PL_price"]

configs = [
    {"model": "DNN_2", "calibration_window": 2, "exog": EPF_EXOG},
    {"model": "DNN_3", "calibration_window": 3, "exog": EPF_EXOG},
]

test_start = dataset_test.index.min()
test_end = dataset_test.index.max()

## 1. Backtest

The tuned hyperparameters stay fixed. For each of the 364 test days the network is retrained on the last `calibration_window` years and forecasts the next 24 hours. A day that fails with an error during training is retried. The forecasts are saved to `results/dnn/test/full/<model>.csv`.

In [ ]:
for config in configs:
    evaluate_dnn(
        dataset=dataset_full,
        calibration_window=config["calibration_window"],
        exog=config["exog"],
        forecast_start=test_start,
        forecast_end=test_end,
        experiment_id=config["model"],
        path=forecast_path(config["model"], split="test", run="full", family="dnn"),
    )

## 2. Backtest comparison

Same metrics as for LEAR: MAE and RMSE in EUR/MWh, and rMAE relative to the weekly naive forecast. Lower is better for all three.

In [4]:
models = [config["model"] for config in configs]
forecasts = load_forecasts(
    [forecast_path(model, split="test", run="full", family="dnn") for model in models]
)

compare_models(
    forecasts,
    prices,
    metrics=["rMAE", "MAE", "RMSE", "sMAPE"],
)

,rMAE,MAE,RMSE,sMAPE
model,,,,
DNN_2,0.618667,22.944370,37.501518,0.287414
DNN_3,0.615529,22.827972,37.371745,0.283859
